In [ ]:
# welcome to my preliminary code analysis for my AP Lang research independent project
# Do same models repeat similar ideas across runs, do different models converge on similar ideas, and do some prompts have narrow idea pools? This is similar to Jiang’s study applied to high school ap lang prompts.
# In a full study, English teachers or human annotators would label each thesis angles and group them, but for simplicity, here I use LLMs.

In [2]:
!pip install openai anthropic google-genai pandas python-dotenv

In [3]:
import os

os.environ["OPENAI_API_KEY"] = "ADD YOUR OWN KEY"
os.environ["ANTHROPIC_API_KEY"] = "ADD YOUR OWN KEY"
os.environ["GEMINI_API_KEY"] = "ADD YOUR OWN KEY"

In [16]:
SYSTEM_PROMPT = """
You are participating in a research study about whether LLMs generate homogeneous thesis angles for AP Language-style writing prompts.

Your task is NOT to write an essay.

For the given prompt, generate exactly 5 thesis angles or argumentative approaches that you believe would fit the prompt well and score high.

Rules:
- Each angle must be one concise sentence.
- Do not write body paragraphs.
- Do not include evidence paragraphs.
- Do not repeat the same basic idea.
- Focus on specific idea argument angles, not polished wording.
- Return only a numbered list from 1 to 5.
"""

In [17]:
%%writefile collect_openai.py

import os
import re
import time
import argparse
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

SYSTEM_PROMPT = """
You are participating in a research study about whether LLMs generate homogeneous thesis angles for AP Language-style writing prompts.

Your task is NOT to write an essay.

For the given prompt, generate exactly 5 distinct thesis angles or argumentative approaches that a high school AP Language student could take.

Rules:
- Each angle must be one concise sentence.
- Do not write body paragraphs.
- Do not include evidence paragraphs.
- Do not repeat the same basic idea.
- Focus on idea-level argument angles, not polished wording.
- Return only a numbered list from 1 to 5.
"""

def parse_numbered_list(text):
    ideas = []
    for line in text.splitlines():
        line = line.strip()
        match = re.match(r"^\s*(\d+)[\).\s-]+(.+)$", line)
        if match:
            ideas.append(match.group(2).strip())
    return ideas[:5]

def call_openai(model, prompt_id, prompt_text, run_id, temperature, max_retries=5):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    user_prompt = f"AP Language prompt:\n\n{prompt_text}"
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.responses.create(
                model=model,
                instructions=SYSTEM_PROMPT,
                input=user_prompt,
                temperature=temperature,
                max_output_tokens=6000,
            )

            raw_text = response.output_text
            ideas = parse_numbered_list(raw_text)

            rows = []
            for idea_rank, idea in enumerate(ideas, start=1):
                output_id = f"{prompt_id}__openai__{model}__run{run_id}__idea{idea_rank}"
                rows.append({
                    "output_id": output_id,
                    "prompt_id": prompt_id,
                    "provider": "openai",
                    "model": model,
                    "run_id": run_id,
                    "idea_rank": idea_rank,
                    "raw_idea": idea
                })

            return {
                "ok": True,
                "prompt_id": prompt_id,
                "model": model,
                "run_id": run_id,
                "rows": rows,
                "raw_text": raw_text,
                "parsed_count": len(ideas)
            }

        except Exception as e:
            last_error = e
            wait = min(60, 2 ** attempt)
            print(f"[OpenAI retry] {prompt_id} | {model} | run {run_id} | attempt {attempt} failed: {repr(e)}")
            print(f"Sleeping {wait}s...")
            time.sleep(wait)

    return {
        "ok": False,
        "prompt_id": prompt_id,
        "model": model,
        "run_id": run_id,
        "error": repr(last_error),
        "rows": []
    }

def append_rows(rows, out_file):
    df_new = pd.DataFrame(rows)
    if Path(out_file).exists():
        df_old = pd.read_csv(out_file)
        df = pd.concat([df_old, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=["output_id"], keep="last")
    else:
        df = df_new

    df.to_csv(out_file, index=False)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--runs", type=int, default=3)
    parser.add_argument("--temperature", type=float, default=1.0)
    parser.add_argument("--prompts", default="prompts.csv")
    parser.add_argument("--out", default="llm_outputs.csv")
    parser.add_argument("--max-workers", type=int, default=100)
    parser.add_argument("--max-retries", type=int, default=5)
    args = parser.parse_args()

    prompts = pd.read_csv(args.prompts)

    tasks = []
    for _, row in prompts.iterrows():
        prompt_id = str(row["prompt_id"]).strip()
        prompt_text = str(row["prompt_text"]).strip()

        if not prompt_text or prompt_text.lower() == "nan":
            print(f"Skipping empty prompt: {prompt_id}")
            continue

        for run_id in range(1, args.runs + 1):
            tasks.append((args.model, prompt_id, prompt_text, run_id, args.temperature, args.max_retries))

    print(f"Starting OpenAI collection")
    print(f"Model: {args.model}")
    print(f"Tasks: {len(tasks)}")
    print(f"Max workers: {args.max_workers}")

    all_rows = []
    failures = []

    with ThreadPoolExecutor(max_workers=args.max_workers) as executor:
        future_to_task = {
            executor.submit(call_openai, *task): task
            for task in tasks
        }

        for i, future in enumerate(as_completed(future_to_task), start=1):
            result = future.result()

            if result["ok"]:
                all_rows.extend(result["rows"])
                status = "OK" if result["parsed_count"] == 5 else f"WARNING parsed {result['parsed_count']}/5"
                print(f"[{i}/{len(tasks)}] {status}: {result['prompt_id']} | {result['model']} | run {result['run_id']}")
                if result["parsed_count"] != 5:
                    print(result["raw_text"])
            else:
                failures.append(result)
                print(f"[{i}/{len(tasks)}] FAILED: {result['prompt_id']} | {result['model']} | run {result['run_id']} | {result['error']}")

    append_rows(all_rows, args.out)

    if failures:
        pd.DataFrame(failures).to_csv(f"failures_openai_{args.model}.csv", index=False)
        print(f"Saved failures to failures_openai_{args.model}.csv")

    print(f"Appended {len(all_rows)} rows to {args.out}")

if __name__ == "__main__":
    main()

Overwriting collect_openai.py


In [18]:
%%writefile collect_claude.py

import os
import re
import time
import argparse
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

SYSTEM_PROMPT = """
You are participating in a research study about whether LLMs generate homogeneous thesis angles for AP Language-style writing prompts.

Your task is NOT to write an essay.

For the given prompt, generate exactly 5 distinct thesis angles or argumentative approaches that a high school AP Language student could take.

Rules:
- Each angle must be one concise sentence.
- Do not write body paragraphs.
- Do not include evidence paragraphs.
- Do not repeat the same basic idea.
- Focus on idea-level argument angles, not polished wording.
- Return only a numbered list from 1 to 5.
"""

def parse_numbered_list(text):
    ideas = []
    for line in text.splitlines():
        line = line.strip()
        match = re.match(r"^\s*(\d+)[\).\s-]+(.+)$", line)
        if match:
            ideas.append(match.group(2).strip())
    return ideas[:5]

def extract_text(message):
    parts = []
    for block in message.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
    return "\n".join(parts)

def call_claude(model, prompt_id, prompt_text, run_id, temperature, max_retries=5):
    client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    user_prompt = f"AP Language prompt:\n\n{prompt_text}"
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            message = client.messages.create(
                model=model,
                max_tokens=5000,
                temperature=temperature,
                system=SYSTEM_PROMPT,
                messages=[
                    {"role": "user", "content": user_prompt}
                ],
            )

            raw_text = extract_text(message)
            ideas = parse_numbered_list(raw_text)

            rows = []
            for idea_rank, idea in enumerate(ideas, start=1):
                output_id = f"{prompt_id}__anthropic__{model}__run{run_id}__idea{idea_rank}"
                rows.append({
                    "output_id": output_id,
                    "prompt_id": prompt_id,
                    "provider": "anthropic",
                    "model": model,
                    "run_id": run_id,
                    "idea_rank": idea_rank,
                    "raw_idea": idea
                })

            return {
                "ok": True,
                "prompt_id": prompt_id,
                "model": model,
                "run_id": run_id,
                "rows": rows,
                "raw_text": raw_text,
                "parsed_count": len(ideas)
            }

        except Exception as e:
            last_error = e
            wait = min(60, 2 ** attempt)
            print(f"[Claude retry] {prompt_id} | {model} | run {run_id} | attempt {attempt} failed: {repr(e)}")
            print(f"Sleeping {wait}s...")
            time.sleep(wait)

    return {
        "ok": False,
        "prompt_id": prompt_id,
        "model": model,
        "run_id": run_id,
        "error": repr(last_error),
        "rows": []
    }

def append_rows(rows, out_file):
    df_new = pd.DataFrame(rows)
    if Path(out_file).exists():
        df_old = pd.read_csv(out_file)
        df = pd.concat([df_old, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=["output_id"], keep="last")
    else:
        df = df_new

    df.to_csv(out_file, index=False)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--runs", type=int, default=3)
    parser.add_argument("--temperature", type=float, default=1.0)
    parser.add_argument("--prompts", default="prompts.csv")
    parser.add_argument("--out", default="llm_outputs.csv")
    parser.add_argument("--max-workers", type=int, default=5)
    parser.add_argument("--max-retries", type=int, default=5)
    args = parser.parse_args()

    prompts = pd.read_csv(args.prompts)

    tasks = []
    for _, row in prompts.iterrows():
        prompt_id = str(row["prompt_id"]).strip()
        prompt_text = str(row["prompt_text"]).strip()

        if not prompt_text or prompt_text.lower() == "nan":
            print(f"Skipping empty prompt: {prompt_id}")
            continue

        for run_id in range(1, args.runs + 1):
            tasks.append((args.model, prompt_id, prompt_text, run_id, args.temperature, args.max_retries))

    print(f"Starting Claude collection")
    print(f"Model: {args.model}")
    print(f"Tasks: {len(tasks)}")
    print(f"Max workers: {args.max_workers}")

    all_rows = []
    failures = []

    with ThreadPoolExecutor(max_workers=args.max_workers) as executor:
        future_to_task = {
            executor.submit(call_claude, *task): task
            for task in tasks
        }

        for i, future in enumerate(as_completed(future_to_task), start=1):
            result = future.result()

            if result["ok"]:
                all_rows.extend(result["rows"])
                status = "OK" if result["parsed_count"] == 5 else f"WARNING parsed {result['parsed_count']}/5"
                print(f"[{i}/{len(tasks)}] {status}: {result['prompt_id']} | {result['model']} | run {result['run_id']}")
                if result["parsed_count"] != 5:
                    print(result["raw_text"])
            else:
                failures.append(result)
                print(f"[{i}/{len(tasks)}] FAILED: {result['prompt_id']} | {result['model']} | run {result['run_id']} | {result['error']}")

    append_rows(all_rows, args.out)

    if failures:
        pd.DataFrame(failures).to_csv(f"failures_claude_{args.model}.csv", index=False)
        print(f"Saved failures to failures_claude_{args.model}.csv")

    print(f"Appended {len(all_rows)} rows to {args.out}")

if __name__ == "__main__":
    main()

Overwriting collect_claude.py


In [19]:
%%writefile collect_gemini.py

import os
import re
import time
import argparse
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

SYSTEM_PROMPT = """
You are participating in a research study about whether LLMs generate homogeneous thesis angles for AP Language-style writing prompts.

Your task is NOT to write an essay.

For the given prompt, generate exactly 5 distinct thesis angles or argumentative approaches that a high school AP Language student could take.

Rules:
- Each angle must be one concise sentence.
- Do not write body paragraphs.
- Do not include evidence paragraphs.
- Do not repeat the same basic idea.
- Focus on idea-level argument angles, not polished wording.
- Return only a numbered list from 1 to 5.
"""

def parse_numbered_list(text):
    ideas = []
    for line in text.splitlines():
        line = line.strip()
        match = re.match(r"^\s*(\d+)[\).\s-]+(.+)$", line)
        if match:
            ideas.append(match.group(2).strip())
    return ideas[:5]

def call_gemini(model, prompt_id, prompt_text, run_id, temperature, max_retries=5):
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    user_prompt = f"{SYSTEM_PROMPT}\n\nAP Language prompt:\n\n{prompt_text}"
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=model,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    temperature=temperature,
                    max_output_tokens=5000,
                )
            )

            raw_text = response.text or ""
            ideas = parse_numbered_list(raw_text)

            rows = []
            for idea_rank, idea in enumerate(ideas, start=1):
                output_id = f"{prompt_id}__google__{model}__run{run_id}__idea{idea_rank}"
                rows.append({
                    "output_id": output_id,
                    "prompt_id": prompt_id,
                    "provider": "google",
                    "model": model,
                    "run_id": run_id,
                    "idea_rank": idea_rank,
                    "raw_idea": idea
                })

            return {
                "ok": True,
                "prompt_id": prompt_id,
                "model": model,
                "run_id": run_id,
                "rows": rows,
                "raw_text": raw_text,
                "parsed_count": len(ideas)
            }

        except Exception as e:
            last_error = e
            wait = min(60, 2 ** attempt)
            print(f"[Gemini retry] {prompt_id} | {model} | run {run_id} | attempt {attempt} failed: {repr(e)}")
            print(f"Sleeping {wait}s...")
            time.sleep(wait)

    return {
        "ok": False,
        "prompt_id": prompt_id,
        "model": model,
        "run_id": run_id,
        "error": repr(last_error),
        "rows": []
    }

def append_rows(rows, out_file):
    df_new = pd.DataFrame(rows)
    if Path(out_file).exists():
        df_old = pd.read_csv(out_file)
        df = pd.concat([df_old, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=["output_id"], keep="last")
    else:
        df = df_new

    df.to_csv(out_file, index=False)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", required=True)
    parser.add_argument("--runs", type=int, default=3)
    parser.add_argument("--temperature", type=float, default=1.0)
    parser.add_argument("--prompts", default="prompts.csv")
    parser.add_argument("--out", default="llm_outputs.csv")
    parser.add_argument("--max-workers", type=int, default=5)
    parser.add_argument("--max-retries", type=int, default=5)
    args = parser.parse_args()

    prompts = pd.read_csv(args.prompts)

    tasks = []
    for _, row in prompts.iterrows():
        prompt_id = str(row["prompt_id"]).strip()
        prompt_text = str(row["prompt_text"]).strip()

        if not prompt_text or prompt_text.lower() == "nan":
            print(f"Skipping empty prompt: {prompt_id}")
            continue

        for run_id in range(1, args.runs + 1):
            tasks.append((args.model, prompt_id, prompt_text, run_id, args.temperature, args.max_retries))

    print(f"Starting Gemini collection")
    print(f"Model: {args.model}")
    print(f"Tasks: {len(tasks)}")
    print(f"Max workers: {args.max_workers}")

    all_rows = []
    failures = []

    with ThreadPoolExecutor(max_workers=args.max_workers) as executor:
        future_to_task = {
            executor.submit(call_gemini, *task): task
            for task in tasks
        }

        for i, future in enumerate(as_completed(future_to_task), start=1):
            result = future.result()

            if result["ok"]:
                all_rows.extend(result["rows"])
                status = "OK" if result["parsed_count"] == 5 else f"WARNING parsed {result['parsed_count']}/5"
                print(f"[{i}/{len(tasks)}] {status}: {result['prompt_id']} | {result['model']} | run {result['run_id']}")
                if result["parsed_count"] != 5:
                    print(result["raw_text"])
            else:
                failures.append(result)
                print(f"[{i}/{len(tasks)}] FAILED: {result['prompt_id']} | {result['model']} | run {result['run_id']} | {result['error']}")

    append_rows(all_rows, args.out)

    if failures:
        pd.DataFrame(failures).to_csv(f"failures_gemini_{args.model}.csv", index=False)
        print(f"Saved failures to failures_gemini_{args.model}.csv")

    print(f"Appended {len(all_rows)} rows to {args.out}")

if __name__ == "__main__":
    main()

Overwriting collect_gemini.py


In [20]:
# CLUSTERING SCRIPT

In [2]:
%%writefile cluster_ideas_by_prompt_openai.py

import os
import time
import argparse
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()

CLUSTERING_SYSTEM_PROMPT = """
You are helping code research data for a study on LLM-generated thesis-angle homogeneity.

You will receive thesis angles generated for ONE AP Language prompt.

Your job:
- Group ideas into BROAD argumentative buckets, not overly specific subcategories.
- The goal is to identify whether outputs converge around the same general thesis angle.
- Prefer fewer broader clusters over many narrow clusters.
- If two ideas would lead to basically the same AP Lang essay thesis, place them in the same cluster.
- Treat minor wording differences and synonyms as the same idea.
- Keep ideas separate only if they would lead to meaningfully different essays.

Return TSV only. Do not use markdown. Do not use code fences. Do not include explanations.

Each output row must have exactly 3 tab-separated columns:
idea_key    canonical_idea    cluster_id

Rules:
- Include one row for every idea_key provided.
- Copy idea_key exactly.
- canonical_idea should be a broad standardized thesis angle.
- cluster_id should be a short lowercase snake_case label.
- Do not include a header row.
"""

def parse_tsv_output(text):
    rows = []
    bad_lines = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue

        parts = line.split("\t")

        if len(parts) != 3:
            bad_lines.append(line)
            continue

        idea_key, canonical_idea, cluster_id = parts

        rows.append({
            "idea_key": idea_key.strip(),
            "canonical_idea": canonical_idea.strip(),
            "cluster_id": cluster_id.strip()
        })

    return rows, bad_lines

def call_cluster_model(model, prompt_text, idea_records, max_retries=5, max_output_tokens=8000):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

    idea_list = "\n".join([
        f"{rec['idea_key']}: {rec['raw_idea']}"
        for rec in idea_records
    ])

    user_prompt = f"""
AP Language prompt:
{prompt_text}

Cluster the following thesis-angle ideas for this prompt.

Important:
- Return one TSV row for each idea_key.
- Do not copy the raw idea back.
- Use tabs between the three columns.

Ideas:
{idea_list}
"""

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(f"    API attempt {attempt}/{max_retries}...")

            response = client.responses.create(
                model=model,
                instructions=CLUSTERING_SYSTEM_PROMPT,
                input=user_prompt,
                max_output_tokens=max_output_tokens,
            )

            mappings, bad_lines = parse_tsv_output(response.output_text)

            if not mappings:
                raise ValueError(f"No valid TSV rows parsed. Raw output start: {response.output_text[:1000]}")

            return {
                "mappings": mappings,
                "bad_lines": bad_lines,
                "raw_output": response.output_text
            }

        except Exception as e:
            last_error = e
            err = str(e).lower()

            if "model_not_found" in err or "does not exist" in err:
                raise RuntimeError(f"Non-retryable model error: {repr(e)}")

            wait_seconds = min(60, 2 ** attempt)
            print(f"    Attempt {attempt} failed: {repr(e)}")
            print(f"    Backing off for {wait_seconds} seconds...")
            time.sleep(wait_seconds)

    raise RuntimeError(f"Clustering failed after {max_retries} attempts: {repr(last_error)}")

def build_cluster_preview(mappings):
    if not mappings:
        return "    No mappings returned."

    temp_df = pd.DataFrame(mappings)

    lines = ["    Cluster preview:"]

    for cluster_id, group in temp_df.groupby("cluster_id"):
        canonical = group["canonical_idea"].iloc[0]
        count = len(group)
        lines.append(f"      - {cluster_id}: {count} idea(s) | {canonical}")

    return "\n".join(lines)

def cluster_one_prompt(idx, total_prompts, prompt_id, group, prompt_lookup, args):
    prompt_id = str(prompt_id)
    prompt_text = prompt_lookup.get(prompt_id, "")

    unique_df = (
        group[["raw_idea"]]
        .dropna()
        .astype(str)
        .apply(lambda col: col.str.strip())
        .drop_duplicates()
        .reset_index(drop=True)
    )

    idea_records = []
    for i, row in unique_df.iterrows():
        idea_records.append({
            "idea_key": f"{prompt_id}_idea_{i+1}",
            "raw_idea": row["raw_idea"]
        })

    print("=" * 80)
    print(f"Starting prompt {idx}/{total_prompts}: {prompt_id}")
    print(f"Rows for prompt: {len(group)}")
    print(f"Unique ideas to cluster: {len(idea_records)}")

    if len(idea_records) == 0:
        return {
            "ok": True,
            "prompt_id": prompt_id,
            "mappings": [],
            "preview": "No ideas to cluster.",
            "bad_lines": []
        }

    result = call_cluster_model(
        model=args.model,
        prompt_text=prompt_text,
        idea_records=idea_records,
        max_retries=args.max_retries,
        max_output_tokens=args.max_output_tokens,
    )

    mappings = result.get("mappings", [])
    bad_lines = result.get("bad_lines", [])

    idea_lookup = {
        rec["idea_key"]: rec["raw_idea"]
        for rec in idea_records
    }

    cleaned_mappings = []

    for m in mappings:
        idea_key = str(m.get("idea_key", "")).strip()

        if idea_key not in idea_lookup:
            continue

        cleaned_mappings.append({
            "prompt_id": prompt_id,
            "idea_key": idea_key,
            "raw_idea": idea_lookup[idea_key],
            "canonical_idea": str(m.get("canonical_idea", "")).strip(),
            "cluster_id": str(m.get("cluster_id", "")).strip()
        })

    returned_keys = {m["idea_key"] for m in cleaned_mappings}
    expected_keys = {rec["idea_key"] for rec in idea_records}
    missing_keys = sorted(expected_keys - returned_keys)

    preview = build_cluster_preview(cleaned_mappings)

    if bad_lines:
        preview += f"\n    Warning: {len(bad_lines)} bad TSV lines were skipped."
        preview += "\n    First few bad lines:"
        for line in bad_lines[:5]:
            preview += f"\n      - {line[:200]}"

    if missing_keys:
        preview += f"\n    Warning: {len(missing_keys)} idea keys were not mapped."
        preview += "\n    First few missing keys:"
        for key in missing_keys[:5]:
            preview += f"\n      - {key}: {idea_lookup[key][:160]}"

    return {
        "ok": True,
        "prompt_id": prompt_id,
        "mappings": cleaned_mappings,
        "preview": preview,
        "unique_ideas": len(idea_records),
        "returned_mappings": len(cleaned_mappings),
        "bad_lines": bad_lines
    }

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--outputs", default="llm_outputs.csv")
    parser.add_argument("--prompts", default="prompts.csv")
    parser.add_argument("--model", default="gpt-5-mini")
    parser.add_argument("--map-out", default="idea_cluster_map.csv")
    parser.add_argument("--coded-out", default="coded_ideas.csv")
    parser.add_argument("--max-retries", type=int, default=5)
    parser.add_argument("--max-output-tokens", type=int, default=8000)
    parser.add_argument("--max-workers", type=int, default=10)
    args = parser.parse_args()

    print("Loading files...")
    outputs = pd.read_csv(args.outputs)
    prompts = pd.read_csv(args.prompts)

    required_outputs = {"output_id", "prompt_id", "provider", "model", "run_id", "idea_rank", "raw_idea"}
    missing_outputs = required_outputs - set(outputs.columns)
    if missing_outputs:
        raise ValueError(f"Missing required columns in outputs file: {missing_outputs}")

    required_prompts = {"prompt_id", "prompt_text"}
    missing_prompts = required_prompts - set(prompts.columns)
    if missing_prompts:
        raise ValueError(f"Missing required columns in prompts file: {missing_prompts}")

    prompt_lookup = dict(zip(prompts["prompt_id"].astype(str), prompts["prompt_text"].astype(str)))

    prompt_groups = list(outputs.groupby("prompt_id"))
    total_prompts = len(prompt_groups)

    print(f"Found {len(outputs)} total output rows.")
    print(f"Found {total_prompts} prompts to cluster.")
    print(f"Using clustering model: {args.model}")
    print(f"Max workers: {args.max_workers}")
    print()

    all_mappings = []
    failures = []
    all_bad_lines = []

    with ThreadPoolExecutor(max_workers=args.max_workers) as executor:
        future_to_prompt = {}

        for idx, (prompt_id, group) in enumerate(prompt_groups, start=1):
            future = executor.submit(
                cluster_one_prompt,
                idx,
                total_prompts,
                prompt_id,
                group.copy(),
                prompt_lookup,
                args,
            )
            future_to_prompt[future] = str(prompt_id)

        for completed_idx, future in enumerate(as_completed(future_to_prompt), start=1):
            prompt_id = future_to_prompt[future]

            try:
                result = future.result()

                mappings = result.get("mappings", [])
                all_mappings.extend(mappings)

                for line in result.get("bad_lines", []):
                    all_bad_lines.append({
                        "prompt_id": prompt_id,
                        "bad_line": line
                    })

                print("=" * 80)
                print(f"Finished {completed_idx}/{total_prompts}: {prompt_id}")
                print(f"Returned mappings: {len(mappings)}")
                print(result.get("preview", ""))

            except Exception as e:
                failures.append({
                    "prompt_id": prompt_id,
                    "error": repr(e)
                })
                print("=" * 80)
                print(f"FAILED {completed_idx}/{total_prompts}: {prompt_id}")
                print(repr(e))

    if failures:
        pd.DataFrame(failures).to_csv("cluster_failures.csv", index=False)
        print("Saved failures to cluster_failures.csv")

    if all_bad_lines:
        pd.DataFrame(all_bad_lines).to_csv("bad_cluster_lines.csv", index=False)
        print("Saved bad TSV lines to bad_cluster_lines.csv")

    print("=" * 80)
    print("Building final coded file...")

    map_df = pd.DataFrame(all_mappings)

    if map_df.empty:
        raise ValueError("No mappings were created. Check cluster_failures.csv.")

    required_map = {"prompt_id", "raw_idea", "canonical_idea", "cluster_id"}
    if not required_map.issubset(set(map_df.columns)):
        raise ValueError(f"Cluster output missing required columns. Got: {map_df.columns}")

    for col in ["prompt_id", "raw_idea", "canonical_idea", "cluster_id"]:
        map_df[col] = map_df[col].astype(str).str.strip()

    map_df.to_csv(args.map_out, index=False)

    outputs["prompt_id"] = outputs["prompt_id"].astype(str).str.strip()
    outputs["raw_idea_clean"] = outputs["raw_idea"].astype(str).str.strip()

    coded = outputs.merge(
        map_df,
        left_on=["prompt_id", "raw_idea_clean"],
        right_on=["prompt_id", "raw_idea"],
        how="left",
        suffixes=("", "_map")
    )

    coded.drop(columns=["raw_idea_clean", "raw_idea_map"], inplace=True, errors="ignore")

    missing = coded["canonical_idea"].isna().sum()
    if missing:
        print(f"Warning: {missing} output rows did not receive a canonical idea.")
        missing_rows = coded[coded["canonical_idea"].isna()][["prompt_id", "model", "run_id", "idea_rank", "raw_idea"]]
        missing_rows.to_csv("missing_cluster_mappings.csv", index=False)
        print("Saved missing rows to missing_cluster_mappings.csv")

    coded.to_csv(args.coded_out, index=False)

    print("Done.")
    print(f"Saved mapping to {args.map_out}")
    print(f"Saved coded data to {args.coded_out}")

if __name__ == "__main__":
    main()

Overwriting cluster_ideas_by_prompt_openai.py


In [5]:
%%writefile analyze_clusters.py

import pandas as pd
from itertools import combinations
import math

INPUT = "coded_ideas.csv"

df = pd.read_csv(INPUT)

required = ["prompt_id", "provider", "model", "run_id", "canonical_idea", "cluster_id"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["prompt_id"] = df["prompt_id"].astype(str)
df["provider"] = df["provider"].astype(str)
df["model"] = df["model"].astype(str)
df["cluster_id"] = df["cluster_id"].astype(str)
df["canonical_idea"] = df["canonical_idea"].astype(str)

def entropy(labels):
    counts = labels.value_counts()
    total = counts.sum()
    h = 0
    for count in counts:
        p = count / total
        h -= p * math.log2(p)
    return h

# ----------------------------
# 1. Intra-model repetition
# ----------------------------

intra_results = []

for (prompt_id, model), group in df.groupby(["prompt_id", "model"]):
    total = len(group)
    unique_clusters = group["cluster_id"].nunique()
    repetition_rate = 1 - unique_clusters / total if total else 0
    top_cluster_share = group["cluster_id"].value_counts(normalize=True).iloc[0]

    top_cluster = group["cluster_id"].value_counts().index[0]
    top_cluster_count = group["cluster_id"].value_counts().iloc[0]

    intra_results.append({
        "prompt_id": prompt_id,
        "model": model,
        "total_ideas": total,
        "unique_clusters": unique_clusters,
        "intra_model_repetition_rate": repetition_rate,
        "top_cluster": top_cluster,
        "top_cluster_count": top_cluster_count,
        "top_cluster_share": top_cluster_share
    })

intra_df = pd.DataFrame(intra_results)
intra_df.to_csv("intra_model_repetition.csv", index=False)

# ----------------------------
# 2. Inter-model similarity
# ----------------------------

inter_results = []

for prompt_id, group in df.groupby("prompt_id"):
    models = sorted(group["model"].unique())

    for model_a, model_b in combinations(models, 2):
        clusters_a = set(group[group["model"] == model_a]["cluster_id"])
        clusters_b = set(group[group["model"] == model_b]["cluster_id"])

        intersection = clusters_a & clusters_b
        union = clusters_a | clusters_b

        jaccard = len(intersection) / len(union) if union else 0

        inter_results.append({
            "prompt_id": prompt_id,
            "model_a": model_a,
            "model_b": model_b,
            "shared_clusters": len(intersection),
            "total_clusters_pair": len(union),
            "jaccard_similarity": jaccard,
            "shared_cluster_names": "; ".join(sorted(intersection))
        })

inter_df = pd.DataFrame(inter_results)
inter_df.to_csv("inter_model_similarity.csv", index=False)

# ----------------------------
# 3. Prompt-level diversity
# ----------------------------

prompt_results = []

for prompt_id, group in df.groupby("prompt_id"):
    total = len(group)
    unique_clusters = group["cluster_id"].nunique()
    h = entropy(group["cluster_id"])
    max_h = math.log2(unique_clusters) if unique_clusters > 1 else 0
    normalized_entropy = h / max_h if max_h > 0 else 0

    top_cluster_counts = group["cluster_id"].value_counts()
    top_cluster = top_cluster_counts.index[0]
    top_cluster_count = top_cluster_counts.iloc[0]
    top_cluster_share = top_cluster_count / total if total else 0

    top_clusters = top_cluster_counts.head(5).to_dict()

    prompt_results.append({
        "prompt_id": prompt_id,
        "total_ideas": total,
        "unique_clusters": unique_clusters,
        "diversity_ratio": unique_clusters / total if total else 0,
        "idea_entropy": h,
        "normalized_idea_entropy": normalized_entropy,
        "top_cluster": top_cluster,
        "top_cluster_count": top_cluster_count,
        "top_cluster_share": top_cluster_share,
        "top_5_clusters": str(top_clusters)
    })

prompt_df = pd.DataFrame(prompt_results)
prompt_df.to_csv("prompt_level_diversity.csv", index=False)

# ----------------------------
# 4. Overall summary numbers
# ----------------------------

total_ideas = len(df)
total_prompts = df["prompt_id"].nunique()
total_models = df["model"].nunique()
total_clusters = df.groupby("prompt_id")["cluster_id"].nunique().sum()

overall_diversity_ratio = total_clusters / total_ideas if total_ideas else 0
overall_intra_repetition = intra_df["intra_model_repetition_rate"].mean()
overall_top_cluster_share = prompt_df["top_cluster_share"].mean()
overall_inter_similarity = inter_df["jaccard_similarity"].mean()

highest_intra = intra_df.sort_values("intra_model_repetition_rate", ascending=False).iloc[0]
lowest_diversity_prompt = prompt_df.sort_values("diversity_ratio", ascending=True).iloc[0]
highest_inter_pair = inter_df.sort_values("jaccard_similarity", ascending=False).iloc[0]

# Find common clusters across providers per prompt
provider_overlap_rows = []

for prompt_id, group in df.groupby("prompt_id"):
    provider_clusters = {}
    for provider, pgroup in group.groupby("provider"):
        provider_clusters[provider] = set(pgroup["cluster_id"])

    if len(provider_clusters) >= 2:
        common = set.intersection(*provider_clusters.values())
        union = set.union(*provider_clusters.values())
        overlap_rate = len(common) / len(union) if union else 0

        provider_overlap_rows.append({
            "prompt_id": prompt_id,
            "providers": ", ".join(sorted(provider_clusters.keys())),
            "common_clusters_across_all_providers": len(common),
            "total_provider_union_clusters": len(union),
            "provider_overlap_rate": overlap_rate,
            "common_cluster_names": "; ".join(sorted(common))
        })

provider_overlap_df = pd.DataFrame(provider_overlap_rows)
provider_overlap_df.to_csv("provider_overlap_summary.csv", index=False)

if not provider_overlap_df.empty:
    strongest_provider_overlap = provider_overlap_df.sort_values("provider_overlap_rate", ascending=False).iloc[0]
else:
    strongest_provider_overlap = None

# ----------------------------
# 5. Print terminal summaries
# ----------------------------

print("\n" + "=" * 90)
print("OVERALL DATASET SUMMARY")
print("=" * 90)
print(f"Total generated ideas analyzed: {total_ideas}")
print(f"Total prompts: {total_prompts}")
print(f"Total models: {total_models}")
print(f"Total prompt-level idea clusters: {total_clusters}")
print(f"Overall diversity ratio: {overall_diversity_ratio:.3f}")
print(f"Average intra-model repetition rate: {overall_intra_repetition:.3f}")
print(f"Average inter-model Jaccard similarity: {overall_inter_similarity:.3f}")
print(f"Average top-cluster share per prompt: {overall_top_cluster_share:.3f}")

print("\n" + "=" * 90)
print("MOST REPETITIVE MODEL/PROMPT CASE")
print("=" * 90)
print(
    f"{highest_intra['model']} produced {int(highest_intra['total_ideas'])} ideas "
    f"for the {highest_intra['prompt_id']} prompt, but they collapsed into only "
    f"{int(highest_intra['unique_clusters'])} distinct clusters."
)
print(f"Intra-model repetition rate: {highest_intra['intra_model_repetition_rate']:.3f}")
print(
    f"Most common cluster: {highest_intra['top_cluster']} "
    f"({int(highest_intra['top_cluster_count'])}/{int(highest_intra['total_ideas'])}, "
    f"{highest_intra['top_cluster_share']:.1%})"
)

print("\n" + "=" * 90)
print("LOWEST PROMPT-LEVEL DIVERSITY")
print("=" * 90)
print(
    f"The {lowest_diversity_prompt['prompt_id']} prompt produced "
    f"{int(lowest_diversity_prompt['total_ideas'])} total ideas but only "
    f"{int(lowest_diversity_prompt['unique_clusters'])} broad clusters."
)
print(f"Diversity ratio: {lowest_diversity_prompt['diversity_ratio']:.3f}")
print(
    f"Most common cluster: {lowest_diversity_prompt['top_cluster']} "
    f"({int(lowest_diversity_prompt['top_cluster_count'])}/{int(lowest_diversity_prompt['total_ideas'])}, "
    f"{lowest_diversity_prompt['top_cluster_share']:.1%})"
)

print("\n" + "=" * 90)
print("HIGHEST INTER-MODEL SIMILARITY PAIR")
print("=" * 90)
print(
    f"For the {highest_inter_pair['prompt_id']} prompt, "
    f"{highest_inter_pair['model_a']} and {highest_inter_pair['model_b']} shared "
    f"{int(highest_inter_pair['shared_clusters'])} clusters out of "
    f"{int(highest_inter_pair['total_clusters_pair'])} total clusters."
)
print(f"Jaccard similarity: {highest_inter_pair['jaccard_similarity']:.3f}")
print(f"Shared clusters: {highest_inter_pair['shared_cluster_names']}")

if strongest_provider_overlap is not None:
    print("\n" + "=" * 90)
    print("STRONGEST PROVIDER-LEVEL OVERLAP")
    print("=" * 90)
    print(
        f"For the {strongest_provider_overlap['prompt_id']} prompt, providers "
        f"({strongest_provider_overlap['providers']}) shared "
        f"{int(strongest_provider_overlap['common_clusters_across_all_providers'])} clusters "
        f"out of {int(strongest_provider_overlap['total_provider_union_clusters'])} total provider-level clusters."
    )
    print(f"Provider overlap rate: {strongest_provider_overlap['provider_overlap_rate']:.3f}")
    print(f"Common clusters: {strongest_provider_overlap['common_cluster_names']}")

print("\n" + "=" * 90)
print("PROMPT-LEVEL SUMMARY TABLE")
print("=" * 90)
print(
    prompt_df[
        [
            "prompt_id",
            "total_ideas",
            "unique_clusters",
            "diversity_ratio",
            "top_cluster",
            "top_cluster_share"
        ]
    ]
    .sort_values("diversity_ratio")
    .to_string(index=False)
)

print("\n" + "=" * 90)
print("COPY-PASTE PAPER SENTENCES")
print("=" * 90)

print(
    f"Intra-model repetition was clear. For instance, {highest_intra['model']} produced "
    f"{int(highest_intra['total_ideas'])} ideas for the {highest_intra['prompt_id']} prompt, "
    f"but they collapsed into only {int(highest_intra['unique_clusters'])} distinct clusters. "
    f"Its intra-model repetition rate was {highest_intra['intra_model_repetition_rate']:.3f}, "
    f"while the average intra-model repetition rate across all model-prompt pairs was "
    f"{overall_intra_repetition:.3f}. This means that a large portion of separate outputs "
    f"repeated similar, narrow claims."
)

print()

print(
    f"Inter-model homogeneity was also visible. For example, for the "
    f"{highest_inter_pair['prompt_id']} prompt, {highest_inter_pair['model_a']} and "
    f"{highest_inter_pair['model_b']} shared {int(highest_inter_pair['shared_clusters'])} "
    f"clusters out of {int(highest_inter_pair['total_clusters_pair'])} total clusters, "
    f"for a Jaccard similarity score of {highest_inter_pair['jaccard_similarity']:.3f}. "
    f"Across all model pairs and prompts, the average inter-model similarity score was "
    f"{overall_inter_similarity:.3f}."
)

print()

print(
    f"At the prompt level, the strongest convergence appeared in the "
    f"{lowest_diversity_prompt['prompt_id']} prompt, where {int(lowest_diversity_prompt['total_ideas'])} "
    f"generated thesis angles collapsed into only {int(lowest_diversity_prompt['unique_clusters'])} "
    f"broad idea clusters. This produced a diversity ratio of "
    f"{lowest_diversity_prompt['diversity_ratio']:.3f}, suggesting that many outputs differed "
    f"in wording while repeating the same general argumentative moves."
)

print()
print("\nSaved files:")
print("- intra_model_repetition.csv")
print("- inter_model_similarity.csv")
print("- prompt_level_diversity.csv")
print("- provider_overlap_summary.csv")

Overwriting analyze_clusters.py


In [6]:
# 1. Collect outputs
# !python collect_openai.py --model gpt-5.1 --runs 3 --max-workers 200
# !python collect_openai.py --model gpt-5-mini --runs 3 --max-workers 200

# !python collect_claude.py --model claude-sonnet-4-6 --runs 3 --max-workers 5
# !python collect_claude.py --model claude-haiku-4-5-20251001 --runs 3 --max-workers 5
# !python collect_gemini.py --model gemini-2.5-pro --runs 3 --max-workers 5
# !python collect_gemini.py --model gemini-2.5-flash --runs 3 --max-workers 5

# # 2. Cluster ideas by prompt
# !python cluster_ideas_by_prompt_openai.py --model gpt-5-mini --max-workers 10 --max-output-tokens 20000

# # 3. Analyze
!python analyze_clusters.py


OVERALL DATASET SUMMARY
Total generated ideas analyzed: 900
Total prompts: 10
Total models: 6
Total prompt-level idea clusters: 53
Overall diversity ratio: 0.059
Average intra-model repetition rate: 0.706
Average inter-model Jaccard similarity: 0.793
Average top-cluster share per prompt: 0.366

MOST REPETITIVE MODEL/PROMPT CASE
claude-sonnet-4-6 produced 15 ideas for the collegeboard5 prompt, but they collapsed into only 2 distinct clusters.
Intra-model repetition rate: 0.867
Most common cluster: nuanced_or_contextual (9/15, 60.0%)

LOWEST PROMPT-LEVEL DIVERSITY
The ferrer3 prompt produced 90 total ideas but only 3 broad clusters.
Diversity ratio: 0.033
Most common cluster: threat_conditioned_on_institutions (35/90, 38.9%)

HIGHEST INTER-MODEL SIMILARITY PAIR
For the collegeboard2 prompt, claude-sonnet-4-6 and gemini-2.5-flash shared 5 clusters out of 5 total clusters.
Jaccard similarity: 1.000
Shared clusters: present_balance; present_for_performers; present_privilege_limited; presen